# DataevalBias Analytics Store Demo

This notebook demonstrates the end-to-end workflow for the DataevalBias capability with the analytics store:

1. Load a dataset
2. Run the bias capability (coverage, balance, diversity)
3. Extract scalar records
4. Write to the analytics store
5. Query results via SQL

## Setup

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core._common.dataeval_bias_capability import DataevalBiasBase, DataevalBiasConfig
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

## Load Dataset

We use the small COCO test dataset included in the repository.

In [ ]:
data_root = Path("../../..")
coco_dir = data_root / "tests" / "data_for_tests" / "coco_resized_val2017"

dataset = CocoDetectionDataset(
    root=str(coco_dir),
    ann_file=str(coco_dir / "instances_val2017_resized_6.json"),
)

print(f"Dataset ID: {dataset.metadata['id']}")
print(f"Images: {len(dataset)}")
print(f"Classes: {dataset.metadata.get('index2label', {})}")

## Run DataevalBias Capability

The bias capability evaluates three dimensions:
- **Coverage**: identifies under-represented images using feature embeddings
- **Balance**: mutual information between metadata factors and class labels
- **Diversity**: how well each metadata factor is sampled across the dataset

In [ ]:
capability = DataevalBiasBase()
run = capability.run(datasets=[dataset], use_cache=False)

print(f"Run UID: {run.run_uid[:24]}...")
print(f"\nCoverage:")
print(f"  Total images: {run.outputs.coverage.total}")
print(f"  Uncovered: {len(run.outputs.coverage.uncovered_indices)}")
print(f"  Radius: {run.outputs.coverage.coverage_radius:.4f}")
print(f"\nBalance present: {run.outputs.balance is not None}")
if run.outputs.balance is not None:
    print(f"  Factors: {len(run.outputs.balance.balance['Balance Score'])}")
    print(f"  Factor names: {run.outputs.balance.balance['Factor'].to_list()}")
print(f"\nDiversity present: {run.outputs.diversity is not None}")
if run.outputs.diversity is not None:
    print(f"  Factors: {len(run.outputs.diversity.factors['Diversity Index'])}")
    print(f"  Factor names: {run.outputs.diversity.factors['Factor'].to_list()}")

## Extract Records

The `extract()` method distills the rich output arrays into flat scalar records suitable for the analytics store.

In [ ]:
records = run.extract()
print(f"Extracted {len(records)} record(s)\n")

record = records[0]
print(f"Table: {record.table_name}")
print(f"Dataset ID: {record.dataset_id}")
print(f"\n--- Coverage ---")
print(f"  Total: {record.coverage_total}")
print(f"  Uncovered count: {record.coverage_uncovered_count}")
print(f"  Uncovered ratio: {record.coverage_uncovered_ratio:.4f}")
print(f"  Radius: {record.coverage_radius:.4f}")
print(f"\n--- Balance ---")
print(f"  Num factors: {record.balance_num_factors}")
print(f"  Mean: {record.balance_mean}")
print(f"  Max: {record.balance_max}")
print(f"  Factors above 0.5: {record.balance_factors_above_05}")
print(f"\n--- Diversity ---")
print(f"  Num factors: {record.diversity_num_factors}")
print(f"  Mean: {record.diversity_mean}")
print(f"  Min: {record.diversity_min}")
print(f"  Factors below 0.4: {record.diversity_factors_below_04}")

## Write to Analytics Store

In [ ]:
store_dir = tempfile.mkdtemp(prefix="bias_demo_")
store = AnalyticsStore(ParquetBackend(store_dir))

store.write([run])

print(f"Store path: {store_dir}")
print(f"Tables: {store.list_tables()}")
print(f"\nBias table schema: {store.describe_table('dataeval_bias')}")

## Query via SQL

All results are now queryable via standard SQL.

In [ ]:
# View all bias records
df = store.query_sql("SELECT * FROM dataeval_bias")
print(df)

In [ ]:
# Query specific fields
df = store.query_sql("""
    SELECT
        dataset_id,
        coverage_uncovered_ratio,
        balance_mean,
        balance_factors_above_05,
        diversity_mean,
        diversity_factors_below_04
    FROM dataeval_bias
""")
print(df)

In [ ]:
# Check the auto-populated runs table
df_runs = store.query_sql("""
    SELECT run_uid, capability_table, entity_type, entity_id
    FROM runs
""")
print(df_runs)

## Cross-Capability JOIN Example

The `dataset_id` field enables direct JOINs with other capability tables. For example, if you also ran `DataevalCleaning` on the same dataset:

```sql
SELECT
    b.dataset_id,
    b.coverage_uncovered_ratio,
    b.balance_mean,
    c.exact_duplicate_ratio,
    c.image_outlier_ratio
FROM dataeval_bias b
JOIN dataeval_cleaning c ON b.dataset_id = c.dataset_id
```

This is possible because both `DataevalBiasRecord` and `DataevalCleaningRecord` include `dataset_id` as a shared key.